# Решения: типы и `apply`

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import pandas as pd


def _find(name: str) -> Path:
    for p in (Path(name), Path(f'../../data/{name}'), Path(f'../data/{name}')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f'{name} не найден — положите slim CSV рядом с ноутбуком')


ORDERS_PATH = _find('orders_slim.csv')
CUSTOMERS_PATH = _find('customers_slim.csv')
PAYMENTS_PATH = _find('payments_slim.csv')

orders = pd.read_csv(ORDERS_PATH, parse_dates=['order_purchase_timestamp'])
if 'order_delivered_customer_date' in orders.columns:
    orders['order_delivered_customer_date'] = pd.to_datetime(
        orders['order_delivered_customer_date'], errors='coerce'
    )
customers = pd.read_csv(CUSTOMERS_PATH)
payments = pd.read_csv(PAYMENTS_PATH)


## Урок. 1-6

In [ ]:
n_orders, n_customers, n_payments = len(orders), len(customers), len(payments)
order_num = orders.select_dtypes(include='number').columns.tolist()
order_cat = orders.select_dtypes(exclude='number').columns.tolist()
payment_num = payments.select_dtypes(include='number').columns.tolist()
payment_cat = payments.select_dtypes(exclude='number').columns.tolist()
orders_pay = orders.merge(payments, on='order_id', how='left')
mean_payment = float(orders_pay['payment_value'].mean())
orders_pay['days_to_deliver'] = orders_pay.apply(
    lambda r: (r['order_delivered_customer_date'] - r['order_purchase_timestamp']).days
    if pd.notna(r['order_delivered_customer_date']) else pd.NA,
    axis=1,
)
mean_days = float(orders_pay['days_to_deliver'].dropna().mean())
orders_pay['payment_bin'] = pd.cut(
    orders_pay['payment_value'], bins=[-1, 100, 300, 1000, 10_000], labels=['small', 'mid', 'big', 'very_big']
)
bin_counts = orders_pay['payment_bin'].value_counts(dropna=False)
STATE_NOTE = (
    'Штат может отражать логистику и платёжные привычки: доля card, средний чек, скорость доставки. '
    'Но это прокси-признак, он может ловить шум и территориальные перекосы, поэтому нужен контроль валидацией.'
)
print(n_orders, n_customers, n_payments)
print(order_num, order_cat)
print(round(mean_payment, 2), round(mean_days, 2))
print(bin_counts)
print(STATE_NOTE)

## ДЗ. 1-4

In [ ]:
min_date = orders['order_purchase_timestamp'].min()
max_date = orders['order_purchase_timestamp'].max()
pay_share = payments['payment_type'].value_counts(normalize=True)
payments['is_card'] = (payments['payment_type'] == 'credit_card').astype(int)
mean_card = float(payments.loc[payments['is_card'] == 1, 'payment_value'].mean())
mean_other = float(payments.loc[payments['is_card'] == 0, 'payment_value'].mean())
APPLY_NOTE = (
    'Через apply удобно считать признак из нескольких столбцов строки, например дни до доставки. '
    'Когда формула работает в рамках одного столбца, лучше векторная операция: она короче и быстрее.'
)
print(min_date, max_date)
print(pay_share.round(3))
print(round(mean_card, 2), round(mean_other, 2))
print(APPLY_NOTE)